Loading the data processed by hdWGCNA in R

In [1]:
import math
import seaborn as sns
import numpy as np
import scanpy as sc
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache

In [2]:
# Selecting the brain region
select_region = "Microglia"

In [3]:
# Loading AnnData object
base_path = Path("/data/scRNA/ABCA/AIBS/AWS/expression_matrices/WMB-10Xv3/20230630/")
expr_path = base_path / f"WMB-10Xv3-{select_region}-raw-wmeta.h5ad"
adata = sc.read_h5ad(expr_path)
adata

AnnData object with n_obs × n_vars = 81472 × 32285
    obs: 'cell_barcode', 'barcoded_cell_sample_label', 'library_label', 'feature_matrix_label', 'entity', 'brain_section_label', 'library_method', 'region_of_interest_acronym', 'donor_label', 'donor_genotype', 'donor_sex', 'dataset_label', 'x', 'y', 'cluster_alias', 'neurotransmitter', 'class', 'subclass', 'supertype', 'cluster', 'neurotransmitter_color', 'class_color', 'subclass_color', 'supertype_color', 'cluster_color', 'region_of_interest_order', 'region_of_interest_color', 'batch'
    var: 'gene_symbol'

In [4]:
# Loading Module Eigengenes dataframe
base_path = Path("/data/scRNA/ABCA/AIBS/AWS/expression_matrices/WMB-10Xv3/20230630/outputs")
modules_df = pd.read_csv(base_path / f"WMB-10Xv3-{select_region}-raw-mc-wgcna-modules.csv", index_col="Unnamed: 0")
modules_df.head()

,gene_name,module,color,kME_MG-M1,kME_grey,kME_MG-M2,kME_MG-M3,kME_MG-M4,kME_MG-M5
ENSMUSG00000004296,ENSMUSG00000004296,MG-M1,turquoise,0.067324,-0.005248,0.069128,-0.008137,0.007815,-0.031117
ENSMUSG00000002985,ENSMUSG00000002985,MG-M1,turquoise,0.189944,0.147565,0.190743,0.233783,0.050853,-0.158240
ENSMUSG00000039109,ENSMUSG00000039109,grey,grey,0.001604,0.012000,0.142080,0.001089,0.063053,0.013650
ENSMUSG00000015568,ENSMUSG00000015568,MG-M2,brown,0.067394,0.009954,0.541998,-0.002288,0.017932,-0.027402
ENSMUSG00000020914,ENSMUSG00000020914,grey,grey,0.018736,-0.008138,0.052446,-0.005465,0.032833,0.016444


In [5]:
# Loading Module Eigengenes dataframe
base_path = Path("/data/scRNA/ABCA/AIBS/AWS/expression_matrices/WMB-10Xv3/20230630/outputs")
MEs_df = pd.read_csv(base_path / f"WMB-10Xv3-{select_region}-raw-mc-wgcna-MEs.csv", index_col="Unnamed: 0")
MEs_df = MEs_df.drop(columns=["grey"])
MEs_df.head()

,blue,green,yellow,turquoise,brown
TGGATCAGTAACACCT-478_A02-0,3.520289,-0.434925,-1.310789,1.821404,0.717890
TCACTCGCAAGTTCGT-215_C01-0,-0.486307,-1.138845,-0.099216,-8.427007,-0.556104
GAGTGAGTCCTTATCA-472_B05-0,-3.545015,0.821186,-1.042361,11.035446,5.529251
CACGGGTTCATTTCCA-216_D01-0,2.866931,-1.964552,0.176167,-7.483160,-0.035609
TTTCACATCCGCACTT-171_C01-0,-1.317700,-2.456663,-0.528557,2.435767,1.356410


In [6]:
# Create a mapping from color to module
color_to_module = modules_df.set_index("color")["module"].to_dict()

# Rename the columns of MEs_df
MEs_df.rename(columns=color_to_module, inplace=True)

# Sort the columns of MEs_df alphabetically
MEs_df = MEs_df.reindex(sorted(MEs_df.columns), axis=1)
MEs_df.head()

,MG-M1,MG-M2,MG-M3,MG-M4,MG-M5
TGGATCAGTAACACCT-478_A02-0,1.821404,0.717890,-0.434925,-1.310789,3.520289
TCACTCGCAAGTTCGT-215_C01-0,-8.427007,-0.556104,-1.138845,-0.099216,-0.486307
GAGTGAGTCCTTATCA-472_B05-0,11.035446,5.529251,0.821186,-1.042361,-3.545015
CACGGGTTCATTTCCA-216_D01-0,-7.483160,-0.035609,-1.964552,0.176167,2.866931
TTTCACATCCGCACTT-171_C01-0,2.435767,1.356410,-2.456663,-0.528557,-1.317700


In [7]:
# Getting the list of modules
modules = MEs_df.columns.tolist()

In [8]:
# Adding MEs to AnnData object
merged_df = pd.merge(adata.obs, MEs_df, left_index=True, right_index=True)
merged_df.head()

,cell_barcode,barcoded_cell_sample_label,library_label,feature_matrix_label,entity,brain_section_label,library_method,region_of_interest_acronym,donor_label,donor_genotype,...,supertype_color,cluster_color,region_of_interest_order,region_of_interest_color,batch,MG-M1,MG-M2,MG-M3,MG-M4,MG-M5
TGGATCAGTAACACCT-478_A02-0,TGGATCAGTAACACCT,478_A02,L8TX_210107_02_H11,WMB-10Xv3-CB,cell,NaN,10Xv3,CB,Snap25-IRES2-Cre;Ai14-556943,Ai14(RCL-tdT)/wt,...,#62CC3D,#651FCC,28,#CC0026,0,1.821404,0.717890,-0.434925,-1.310789,3.520289
TCACTCGCAAGTTCGT-215_C01-0,TCACTCGCAAGTTCGT,215_C01,L8TX_200206_01_D03,WMB-10Xv3-CB,cell,NaN,10Xv3,CB,Slc32a1-IRES-Cre;Ai14-507771,Slc32a1-IRES-Cre/wt;Ai14(RCL-tdT)/wt,...,#62CC3D,#651FCC,28,#CC0026,0,-8.427007,-0.556104,-1.138845,-0.099216,-0.486307
GAGTGAGTCCTTATCA-472_B05-0,GAGTGAGTCCTTATCA,472_B05,L8TX_201217_01_H07,WMB-10Xv3-CB,cell,NaN,10Xv3,CB,Snap25-IRES2-Cre;Ai14-557213,Ai14(RCL-tdT)/wt,...,#62CC3D,#651FCC,28,#CC0026,0,11.035446,5.529251,0.821186,-1.042361,-3.545015
CACGGGTTCATTTCCA-216_D01-0,CACGGGTTCATTTCCA,216_D01,L8TX_200206_01_A04,WMB-10Xv3-CB,cell,NaN,10Xv3,CB,Slc32a1-IRES-Cre;Ai14-507773,Slc32a1-IRES-Cre/wt;Ai14(RCL-tdT)/wt,...,#62CC3D,#651FCC,28,#CC0026,0,-7.483160,-0.035609,-1.964552,0.176167,2.866931
TTTCACATCCGCACTT-171_C01-0,TTTCACATCCGCACTT,171_C01,L8TX_191029_01_E08,WMB-10Xv3-CB,cell,NaN,10Xv3,CB,Snap25-IRES2-Cre;Ai14-493659,Snap25-IRES2-Cre/wt;Ai14(RCL-tdT)/wt,...,#62CC3D,#651FCC,28,#CC0026,0,2.435767,1.356410,-2.456663,-0.528557,-1.317700


## Gene Ontology

In [9]:
# For one module
from gprofiler import GProfiler

# Initialize GProfiler
gp = GProfiler(return_dataframe=True)

# Select genes from a specific module
module_name = "MG-M1"
genes_of_module = modules_df[modules_df["module"] == module_name]["gene_name"].tolist()

# Perform GO enrichment analysis
go_results = gp.profile(organism='mmusculus', query=genes_of_module)

# Display the results
go_results.head()

,source,native,name,p_value,significant,description,term_size,query_size,intersection_size,effective_domain_size,precision,recall,query,parents
0,GO:BP,GO:0006950,response to stress,1.238858e-47,True,"""Any process that results in a change in state...",3966,452,198,26963,0.438053,0.049924,query_1,[GO:0050896]
1,GO:BP,GO:0051239,regulation of multicellular organismal process,5.421701e-45,True,"""Any process that modulates the frequency, rat...",3145,452,172,26963,0.380531,0.054690,query_1,"[GO:0032501, GO:0050789]"
2,GO:BP,GO:0002376,immune system process,2.856283e-44,True,"""Any process involved in the development or fu...",2866,452,163,26963,0.360619,0.056874,query_1,[GO:0008150]
3,GO:BP,GO:0002682,regulation of immune system process,1.935714e-43,True,"""Any process that modulates the frequency, rat...",1590,452,121,26963,0.267699,0.076101,query_1,"[GO:0002376, GO:0050789]"
4,GO:BP,GO:0048518,positive regulation of biological process,1.312801e-41,True,"""Any process that activates or increases the f...",6584,452,249,26963,0.550885,0.037819,query_1,"[GO:0008150, GO:0050789]"


In [ ]:
go_results

,source,native,name,p_value,significant,description,term_size,query_size,intersection_size,effective_domain_size,precision,recall,query,parents
0,GO:BP,GO:0006950,response to stress,1.238858e-47,True,"""Any process that results in a change in state...",3966,452,198,26963,0.438053,0.049924,query_1,[GO:0050896]
1,GO:BP,GO:0051239,regulation of multicellular organismal process,5.421701e-45,True,"""Any process that modulates the frequency, rat...",3145,452,172,26963,0.380531,0.054690,query_1,"[GO:0032501, GO:0050789]"
2,GO:BP,GO:0002376,immune system process,2.856283e-44,True,"""Any process involved in the development or fu...",2866,452,163,26963,0.360619,0.056874,query_1,[GO:0008150]
3,GO:BP,GO:0002682,regulation of immune system process,1.935714e-43,True,"""Any process that modulates the frequency, rat...",1590,452,121,26963,0.267699,0.076101,query_1,"[GO:0002376, GO:0050789]"
4,GO:BP,GO:0048518,positive regulation of biological process,1.312801e-41,True,"""Any process that activates or increases the f...",6584,452,249,26963,0.550885,0.037819,query_1,"[GO:0008150, GO:0050789]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1323,KEGG,KEGG:05143,African trypanosomiasis,4.950072e-02,True,African trypanosomiasis,37,257,6,9330,0.023346,0.162162,query_1,[KEGG:00000]
1324,GO:MF,GO:0044183,protein folding chaperone,4.955484e-02,True,"""Binding to a protein or a protein-containing ...",60,452,7,25071,0.015487,0.116667,query_1,[GO:0003674]
1325,GO:CC,GO:0097433,dense body,4.966568e-02,True,"""An electron dense body which may contain gran...",7,458,3,27195,0.006550,0.428571,query_1,"[GO:0005737, GO:0110165]"
1326,GO:BP,GO:0060538,skeletal muscle organ development,4.984587e-02,True,"""The progression of a skeletal muscle organ ov...",219,452,14,26963,0.030973,0.063927,query_1,[GO:0007517]


In [11]:
# For all modules
from gprofiler import GProfiler

for module_name in modules:
    # Initialize GProfiler
    gp = GProfiler(return_dataframe=True)
    
    # Select genes from a specific module
    genes_of_module = modules_df[modules_df["module"] == module_name]["gene_name"].tolist()

    # Perform GO enrichment analysis
    go_results = gp.profile(organism='mmusculus', query=genes_of_module)

    # Display the results
    go_results.to_csv(f"/data/scRNA/ABCA/AIBS/AWS/expression_matrices/WMB-10Xv3/20230630/outputs/GO/{module_name}_GO.csv")
    print(f"Module saved: {module_name}")

Module saved: MG-M1
Module saved: MG-M2
Module saved: MG-M3
Module saved: MG-M4
Module saved: MG-M5
